# 5. Xuất Dữ Liệu sang MongoDB

Notebook này trình bày quá trình xuất dữ liệu đã xử lý từ HDFS sang MongoDB để phục vụ cho Dashboard web.

**Tại sao cần MongoDB?**
- HDFS + Parquet phù hợp cho xử lý batch lớn (ETL, ML) nhưng **KHÔNG** phù hợp cho truy vấn real-time
- MongoDB là NoSQL database, cho phép truy vấn nhanh theo key, phù hợp cho API backend
- Dashboard cần response time < 1 giây, không thể đọc trực tiếp từ HDFS mỗi lần request

> 📌 Notebook này tương ứng với file `spark_jobs/export_to_mongo.py` trong project.

## 5.1 Kiến trúc hệ thống

```
CSV → HDFS (Bronze) → PySpark ETL → HDFS (Silver/Gold) → MongoDB → Flask API → Dashboard
         ↓                                    ↓
    Lưu trữ lâu dài              Phục vụ truy vấn nhanh
```

Mỗi tầng có vai trò riêng:
- **HDFS**: Kho dữ liệu trung tâm, lưu dữ liệu lớn, xử lý batch
- **MongoDB**: Cơ sở dữ liệu phục vụ ứng dụng, truy vấn nhanh (<100ms)
- **Flask API**: Cầu nối giữa MongoDB và Frontend
- **Dashboard**: Giao diện hiển thị cho người dùng

## 5.2 Import thư viện

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/Admin/AppData/Local/Programs/Python/Python313/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ["PYSPARK_PYTHON"]
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pymongo import MongoClient
import json

spark = (
    SparkSession.builder.appName("Olist_Export").master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("✅ SparkSession đã khởi tạo!")

## 5.3 Đọc dữ liệu từ HDFS

Đọc các file Parquet đã được ETL xử lý.

In [ ]:
HDFS_SILVER = "hdfs://localhost:9000/user/bigdata/olist/silver"
HDFS_GOLD = "hdfs://localhost:9000/user/bigdata/olist/gold"

# Đọc merged_orders (Silver)
merged = spark.read.parquet(f"{HDFS_SILVER}/merged_orders")
print(f"📖 merged_orders: {merged.count():,} dòng")

# Đọc rfm_customers (Gold)
rfm = spark.read.parquet(f"{HDFS_GOLD}/rfm_customers")
print(f"📖 rfm_customers: {rfm.count():,} dòng")

## 5.4 Cấu trúc Database MongoDB

### Database: `olist_ecommerce`

| Collection | Mô tả | Số documents |
|-----------|-------|-------------|
| `orders` | Thông tin đơn hàng (denormalized) | ~99,000 |
| `customers` | Khách hàng + RFM + Churn | ~93,000 |
| `products` | Thống kê theo danh mục | ~70 |
| `sellers` | Thống kê theo người bán | ~3,000 |
| `aggregations` | Dữ liệu tổng hợp sẵn cho biểu đồ | ~50 |

### Denormalization (Phi chuẩn hóa)
Trong MongoDB, ta lưu dữ liệu dạng đã join sẵn (denormalized) thay vì chia thành nhiều bảng như SQL. Điều này giúp mỗi query chỉ cần đọc 1 collection thay vì join nhiều bảng.

## 5.5 Tạo Aggregations (Dữ liệu tổng hợp)

### Tại sao cần pre-aggregate?
Thay vì để Dashboard tính toán mỗi lần hiển thị (chậm), ta **tính sẵn** và lưu vào MongoDB:
- Doanh thu theo tháng
- Thống kê theo danh mục sản phẩm
- Phân bố thanh toán
- Thống kê theo bang
- Hiệu suất mô hình ML

Điều này giúp Dashboard load gần như **tức thì** (< 100ms).

In [ ]:
# === Doanh thu theo tháng ===
print("🔧 Tính doanh thu theo tháng...")
monthly_rev = (
    merged
    .groupBy("purchase_year_month")
    .agg(
        F.sum("order_value").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
    )
    .orderBy("purchase_year_month")
).toPandas()

monthly_docs = []
for _, row in monthly_rev.iterrows():
    monthly_docs.append({
        "agg_type": "monthly_revenue",
        "purchase_year_month": row["purchase_year_month"],
        "total_revenue": float(row["total_revenue"]),
        "order_count": int(row["order_count"]),
    })
print(f"   -> {len(monthly_docs)} tháng")

# === Thống kê theo danh mục ===
print("🔧 Tính thống kê theo danh mục...")
cat_stats = (
    merged
    .groupBy("main_category_english")
    .agg(
        F.sum("order_value").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.avg("total_price").alias("avg_order_value"),
        F.avg("review_score").alias("avg_review_score"),
    )
    .orderBy(F.col("total_revenue").desc())
).toPandas()

cat_docs = []
for _, row in cat_stats.iterrows():
    cat_docs.append({
        "agg_type": "category_stats",
        "main_category_english": row["main_category_english"],
        "total_revenue": float(row["total_revenue"]),
        "order_count": int(row["order_count"]),
        "avg_order_value": float(row["avg_order_value"]),
        "avg_review_score": float(row["avg_review_score"]) if row["avg_review_score"] else 0,
    })
print(f"   -> {len(cat_docs)} danh mục")

print("\n✅ Đã tạo xong aggregation documents")

## 5.6 Ghi dữ liệu vào MongoDB

### Chiến lược ghi:
- **drop + insert_many**: Xóa sạch collection cũ rồi ghi mới (đảm bảo không trùng lặp)
- **batch_size = 5000**: Ghi theo lô để tránh timeout
- **Tạo index**: Index trên các field thường query (`order_id`, `customer_unique_id`)

In [ ]:
# Kết nối MongoDB
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "olist_ecommerce"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
print(f"✅ Đã kết nối MongoDB: {MONGO_URI}")
print(f"   Database: {DB_NAME}")

# Ghi orders
print("\n💾 Đang ghi collection 'orders'...")
orders_data = merged.select(
    "order_id", "customer_unique_id", "order_status",
    "order_purchase_timestamp", "total_price", "total_freight_value",
    "order_value", "payment_type", "review_score",
    "main_category_english", "customer_state", "customer_city",
    "delivery_days", "delivery_status", "satisfaction_level",
    "purchase_year_month", "purchase_hour",
).toPandas().to_dict("records")

db["orders"].drop()
BATCH_SIZE = 5000
for i in range(0, len(orders_data), BATCH_SIZE):
    batch = orders_data[i:i+BATCH_SIZE]
    db["orders"].insert_many(batch)
print(f"   ✅ Đã ghi {len(orders_data):,} orders")

# Ghi aggregations
print("\n💾 Đang ghi collection 'aggregations'...")
db["aggregations"].drop()
all_aggs = monthly_docs + cat_docs
db["aggregations"].insert_many(all_aggs)
print(f"   ✅ Đã ghi {len(all_aggs)} aggregation documents")

## 5.7 Tạo Index cho MongoDB

Index giúp tăng tốc truy vấn từ **O(n)** xuống **O(log n)**. Đối với 99,000 documents, index giúp query nhanh hơn **hàng trăm lần**.

In [ ]:
# Tạo index
print("🔧 Tạo index...")
db["orders"].create_index("order_id", unique=True)
db["orders"].create_index("customer_unique_id")
db["orders"].create_index("order_purchase_timestamp")
db["customers"].create_index("customer_unique_id", unique=True)
db["aggregations"].create_index("agg_type")

print("✅ Đã tạo index cho tất cả collection!")

# Xác nhận
print("\n📊 Thống kê MongoDB:")
for col_name in ["orders", "customers", "products", "sellers", "aggregations"]:
    count = db[col_name].count_documents({})
    print(f"   {col_name}: {count:,} documents")

client.close()
print("\n✅ Đã đóng kết nối MongoDB")

## 5.8 Kết luận

- ✅ Đã xuất ~99,000 orders sang MongoDB
- ✅ Đã xuất ~93,000 customers với RFM + Churn data
- ✅ Đã tạo aggregation documents cho Dashboard
- ✅ Đã tạo index để tối ưu truy vấn

### Luồng dữ liệu hoàn chỉnh:
```
📂 CSV (9 files)
  ↓ [01_ingestion.ipynb]
📦 HDFS Bronze (CSV thô)
  ↓ [02_etl.ipynb]
📦 HDFS Silver/Gold (Parquet đã xử lý)
  ↓ [04_ml_models.ipynb] → Mô hình ML
  ↓ [05_export_to_mongo.ipynb - notebook này]
🍃 MongoDB (NoSQL cho truy vấn nhanh)
  ↓
🌐 Flask API (Python backend)
  ↓
📊 Dashboard (HTML/CSS/JS)
```

Dữ liệu giờ đã sẵn sàng cho Flask API và Dashboard hiển thị!

In [ ]:
spark.stop()
print('✅ SparkSession đã dừng.')